# Lesson 9: Your First AI Agent

## From Chatbot to Agent

In this lesson, you'll learn:
- What makes an "agent" different from a chatbot
- The core concepts of agentic AI
- How agents break down and solve problems
- Build a Research Assistant agent!

**Agents = AI that thinks, plans, and takes actions!**


---

## Part 1: Chatbot vs Agent

### What's the Difference?

```
┌─────────────────────────────────────────────────────────────────┐
│ CHATBOT                                                         │
├─────────────────────────────────────────────────────────────────┤
│ • Responds to what you say                                      │
│ • Follows a simple pattern: input → output                      │
│ • You drive the conversation                                    │
│ • Does one thing at a time                                      │
│                                                                 │
│ Example: "What is Python?" → [AI explains Python]               │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ AGENT                                                           │
├─────────────────────────────────────────────────────────────────┤
│ • Understands a goal and works toward it                        │
│ • Can break down complex tasks into steps                       │
│ • Makes decisions about what to do next                         │
│ • Can use "tools" to take actions                               │
│ • Works more autonomously                                       │
│                                                                 │
│ Example: "Research Python for me" →                             │
│   [AI decides what to research]                                 │
│   [AI gathers information]                                      │
│   [AI organizes findings]                                       │
│   [AI presents a report]                                        │
└─────────────────────────────────────────────────────────────────┘
```

### The Agent Loop

```
┌─────────────────────────────────────────────────────────────────┐
│                      THE AGENT LOOP                             │
│                                                                 │
│         ┌──────────┐                                            │
│         │   GOAL   │                                            │
│         └────┬─────┘                                            │
│              │                                                  │
│              ▼                                                  │
│         ┌──────────┐                                            │
│    ┌───→│  THINK   │ ← "What should I do next?"                 │
│    │    └────┬─────┘                                            │
│    │         │                                                  │
│    │         ▼                                                  │
│    │    ┌──────────┐                                            │
│    │    │   ACT    │ ← Take an action or use a tool             │
│    │    └────┬─────┘                                            │
│    │         │                                                  │
│    │         ▼                                                  │
│    │    ┌──────────┐                                            │
│    │    │ OBSERVE  │ ← See the result                           │
│    │    └────┬─────┘                                            │
│    │         │                                                  │
│    │         ▼                                                  │
│    │    ┌──────────┐                                            │
│    └────│  DONE?   │ → If yes, return final answer              │
│         └──────────┘                                            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```


---

## Part 2: Setup


In [ ]:
# Setup - Run this first!

from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
client = OpenAI()

def chat(prompt, temperature=0.7, system=None):
    """Chat with optional system prompt"""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

def get_json(prompt, system=None):
    """Get JSON response"""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("✅ Ready to build an agent!")


---

## Part 3: A Simple Task Decomposition Agent

The first step to being an agent is breaking down complex tasks. Let's build an agent that:
1. Takes a complex task
2. Breaks it into subtasks
3. Solves each subtask
4. Combines the results


In [ ]:
# Simple Task Decomposition Agent

class TaskAgent:
    """An agent that breaks down and solves complex tasks"""
    
    def __init__(self):
        self.task_history = []
    
    def solve(self, task):
        """Solve a complex task by breaking it down"""
        
        print(f"🎯 TASK: {task}")
        print("=" * 60)
        
        # Step 1: THINK - Break down the task
        print("\n🧠 THINKING: Breaking down the task...")
        subtasks = self._decompose_task(task)
        print(f"   Found {len(subtasks)} subtasks:")
        for i, st in enumerate(subtasks, 1):
            print(f"   {i}. {st}")
        
        # Step 2: ACT - Solve each subtask
        print("\n⚡ ACTING: Solving each subtask...")
        results = []
        for i, subtask in enumerate(subtasks, 1):
            print(f"\n   📍 Subtask {i}: {subtask}")
            result = self._solve_subtask(subtask)
            results.append({"subtask": subtask, "result": result})
            print(f"   ✓ Complete")
        
        # Step 3: SYNTHESIZE - Combine results
        print("\n📊 SYNTHESIZING: Combining results...")
        final_answer = self._synthesize(task, results)
        
        print("\n" + "=" * 60)
        print("✅ TASK COMPLETE!")
        
        return final_answer
    
    def _decompose_task(self, task):
        """Break a task into subtasks"""
        prompt = f"""
        Break down this task into 3-4 specific subtasks:
        Task: {task}
        
        Return JSON with:
        {{"subtasks": ["subtask 1", "subtask 2", "subtask 3"]}}
        
        Make each subtask specific and actionable.
        """
        result = get_json(prompt)
        return result["subtasks"]
    
    def _solve_subtask(self, subtask):
        """Solve a single subtask"""
        prompt = f"""
        Complete this subtask thoroughly but concisely (2-3 sentences):
        {subtask}
        """
        return chat(prompt, temperature=0.3)
    
    def _synthesize(self, original_task, results):
        """Combine subtask results into a final answer"""
        results_text = "\n".join([
            f"- {r['subtask']}: {r['result']}" 
            for r in results
        ])
        
        prompt = f"""
        Original task: {original_task}
        
        Subtask results:
        {results_text}
        
        Synthesize these results into a coherent final answer.
        Be comprehensive but concise.
        """
        return chat(prompt, temperature=0.5)

print("✅ TaskAgent created!")


In [ ]:
# Test the agent!

agent = TaskAgent()

result = agent.solve("Explain the benefits and drawbacks of remote work")


In [ ]:
# See the final answer
from IPython.display import Markdown, display

print("📄 FINAL ANSWER:")
print("-" * 40)
display(Markdown(result))


---

## Part 4: Project - Research Assistant Agent

Let's build a more sophisticated agent that researches a topic and creates a structured report!


In [ ]:
# PROJECT: Research Assistant Agent

class ResearchAgent:
    """An agent that researches topics and creates reports"""
    
    def __init__(self):
        self.name = "ResearchBot"
    
    def research(self, topic):
        """Research a topic and create a comprehensive report"""
        
        print(f"🔬 RESEARCH AGENT: Investigating '{topic}'")
        print("=" * 60)
        
        # Phase 1: Understand the topic
        print("\n📚 Phase 1: Understanding the topic...")
        understanding = self._understand_topic(topic)
        print(f"   Key aspects identified: {len(understanding['aspects'])}")
        
        # Phase 2: Generate research questions
        print("\n❓ Phase 2: Generating research questions...")
        questions = self._generate_questions(topic, understanding)
        print(f"   Questions generated: {len(questions)}")
        
        # Phase 3: Answer each question
        print("\n🔍 Phase 3: Researching answers...")
        findings = []
        for i, q in enumerate(questions, 1):
            print(f"   Researching Q{i}: {q[:50]}...")
            answer = self._answer_question(topic, q)
            findings.append({"question": q, "answer": answer})
        
        # Phase 4: Compile report
        print("\n📝 Phase 4: Compiling report...")
        report = self._compile_report(topic, understanding, findings)
        
        print("\n" + "=" * 60)
        print("✅ RESEARCH COMPLETE!")
        
        return report
    
    def _understand_topic(self, topic):
        """Analyze what aspects of the topic to cover"""
        prompt = f"""
        Analyze this research topic: "{topic}"
        
        Return JSON with:
        {{
            "summary": "one sentence summary of the topic",
            "aspects": ["aspect 1", "aspect 2", "aspect 3"],
            "target_audience": "who would benefit from this research"
        }}
        """
        return get_json(prompt)
    
    def _generate_questions(self, topic, understanding):
        """Generate research questions"""
        prompt = f"""
        Generate 3 important research questions about: "{topic}"
        
        Topic aspects to consider: {understanding['aspects']}
        
        Return JSON with:
        {{"questions": ["question 1", "question 2", "question 3"]}}
        
        Make questions specific and answerable.
        """
        result = get_json(prompt)
        return result["questions"]
    
    def _answer_question(self, topic, question):
        """Research and answer a question"""
        prompt = f"""
        Research Topic: {topic}
        Question: {question}
        
        Provide a thorough but concise answer (3-4 sentences).
        Include specific details or examples where helpful.
        """
        return chat(prompt, temperature=0.3)
    
    def _compile_report(self, topic, understanding, findings):
        """Compile everything into a final report"""
        findings_text = "\n\n".join([
            f"**Q: {f['question']}**\n{f['answer']}"
            for f in findings
        ])
        
        prompt = f"""
        Create a well-structured research report:
        
        Topic: {topic}
        Summary: {understanding['summary']}
        Target Audience: {understanding['target_audience']}
        
        Research Findings:
        {findings_text}
        
        Create a report with:
        1. Executive Summary (2 sentences)
        2. Key Findings (bullet points)
        3. Detailed Analysis (based on the Q&A findings)
        4. Conclusion and Recommendations
        
        Format in Markdown.
        """
        return chat(prompt, temperature=0.5)

print("✅ ResearchAgent created!")


In [ ]:
# Use the Research Agent!

researcher = ResearchAgent()
report = researcher.research("The impact of AI on education")


In [ ]:
# Display the research report

print("\n📄 RESEARCH REPORT")
print("=" * 60)
display(Markdown(report))


---

## Summary: Key Takeaways

### Chatbot vs Agent

| Chatbot | Agent |
|---------|-------|
| Reactive | Proactive |
| Single response | Multiple steps |
| You direct | It plans |
| One task | Complex workflows |

### The Agent Pattern

1. **THINK**: Analyze the task, make a plan
2. **ACT**: Execute a step or use a tool
3. **OBSERVE**: Check the result
4. **REPEAT**: Until goal is achieved

### What You Built
- **TaskAgent**: Decomposes and solves complex tasks
- **ResearchAgent**: Autonomously researches topics

### What's Next?
In **Lesson 10**, you'll build your **Final Project** - a complete AI Tutor that teaches any topic!

**You're almost done! Move on to the final lesson!**
